# STUDENT LAB WORKSHEET: End-to-End regresní pipeline a zpracování reálných dat

**Odhadovaná délka cvičení:** 80 minut  
**Potřebné nástroje:** Počítač s webovým prohlížečem, Google účet, přístup k prostředí Google Colab.

## 1. Co se naučíte
Po úspěšném dokončení tohoto cvičení budete schopni:
1. **Identifikovat a zabránit** metodické chybě zvané únik informací (data leakage) správným umístěním funkce pro rozdělení dat.
2. **Sestavit** objekt `ColumnTransformer` pro oddělené a bezpečné předzpracování numerických a kategoriálních příznaků.
3. **Použít** metody `SimpleImputer` a `OneHotEncoder` k algoritmickému ošetření neúplných a textových dat.
4. **Integrovat** kroky předzpracování a samotný model do jediné struktury pomocí třídy `Pipeline`.
5. **Vyhodnotit** přesnost modelu pomocí metriky RMSE a kriticky ji porovnat s teoretickou hranicí šumu v datech.

## 2. Kontext a motivace
Představte si, že pracujete jako datoví analytici a dostali jste za úkol vytvořit model pro predikci platu zaměstnance. K dispozici máte data o délce praxe, věku, bonusech, oddělení a dosaženém vzdělání.

V reálném světě jsou data zřídkakdy čistá. Databáze často obsahují prázdná místa (nevyplněné položky v dotazníku) a textové popisky. Matematické modely, jako je lineární regrese, ale umí pracovat pouze s čísly. V tomto cvičení si ukážeme, jak taková heterogenní (smíšená) a neúplná data automatizovaně a metodicky správně připravit pro strojové učení tak, abyste minimalizovali zkreslení výsledků.

## 3. Klíčové pojmy
* **NaN (Not a Number):** Zástupná hodnota, která v Pythonu reprezentuje chybějící nebo neznámá data.
* **Imputace (Imputation):** Proces algoritmického nahrazení chybějících hodnot (např. průměrem, mediánem nebo nejčastější hodnotou).
* **One-Hot Encoding:** Metoda transformace textových kategorií (např. "HR", "IT") na sérii nových sloupců, které obsahují pouze binární hodnoty 0 a 1.
* **Data Leakage (Únik informací):** Závažná metodická chyba, kdy model získá v trénovací fázi informace z testovacích dat. Vede k falešně optimistickému hodnocení modelu.
* **RMSE (Root Mean Squared Error):** Metrika chybovosti modelu. Udává průměrnou chybu ve stejných jednotkách, jako má cílová proměnná.

---

## 4. Guided lab (Průzkum ukázky)

### Krok A: Analýza baseline modelu
* **Účel:** Pochopit, jak knihovna `scikit-learn` implementuje polynomiální regresi na jednoduchém 1D případu.
* **Instrukce:** Spusťte následující buňku s kódem.
* **Co pozorovat:** Model pro "polynomiální" regresi se trénuje pomocí třídy `LinearRegression()`. Trik spočívá v tom, že data jsou nejprve nelineárně transformována pomocí `PolynomialFeatures`.
* **ÚKOL:** Všimněte si, že do proměnné `mse` se ukládá výsledek `root_mean_squared_error`. Textový výpis (`print`) ale říká "Mean Squared Error". **Opravte text uvnitř funkce `print` tak, aby terminologicky odpovídal skutečně počítané metrice.**


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import root_mean_squared_error

# Set random seed for reproducibility
np.random.seed(42)

# Generate synthetic data: 'YearsExperience' ranges from 0 to 20
n_samples = 100
X_1d = np.linspace(0, 20, n_samples).reshape(-1, 1)

# Define a quadratic relationship for Salary:
base_salary = 20000
coef1 = 1000
coef2 = 200
noise_1d = np.random.normal(0, 1000, size=n_samples)

salary_1d = base_salary + coef1 * X_1d.flatten() + coef2 * X_1d.flatten()**2 + noise_1d

# ----- Polynomial Regression -----
degree = 3
poly_features = PolynomialFeatures(degree=degree)
X_poly = poly_features.fit_transform(X_1d)

model_1d = LinearRegression()
model_1d.fit(X_poly, salary_1d)
salary_pred = model_1d.predict(X_poly)

# Calculate model performance
mse = root_mean_squared_error(salary_1d, salary_pred)
# TODO: Opravte název chybové metriky v následujícím výpisu!
print(f"\nMean Squared Error: {mse:.2f}")

# ----- Plotting the Results -----
plt.scatter(X_1d, salary_1d, color='blue', label='Data')
plt.plot(X_1d, salary_pred, color='red', linewidth=2, label=f'Polynomial Regression (degree={degree})')
plt.xlabel('Years of Experience')
plt.ylabel('Salary')
plt.title('Polynomial Regression Fit to Synthetic Salary Data')
plt.legend()
plt.show()

### Krok B: Generování reálnějších dat a nalezení chyby
* **Účel:** Vygenerovat komplexní dataset pro vaši samostatnou práci a otestovat vaši pozornost k detailům.
* **Instrukce:** Přečtěte si a spusťte kód v následující buňce.
* **Co pozorovat:** Výsledná tabulka `data` obsahuje numerické i textové sloupce a zástupné znaky `NaN`.
* **ÚKOL:** Podívejte se na řádky pod komentářem `# Apply the department multiplier`. Tento kód má aplikovat vliv oddělení (`Department`) na výši platu. Je zde logika správná? **Identifikujte chybu (použití špatného slovníku a zdvojení šumu) a kód opravte.**


In [ ]:
# Generate primary numerical features
n_samples = 1000
years_experience = np.linspace(0, 20, n_samples)
age = np.random.randint(22, 66, size=n_samples)
bonus = np.random.uniform(500, 5000, size=n_samples)

# Categorical features
education_levels = np.random.choice(['Bachelor', 'Master', 'PhD'], size=n_samples, p=[0.5, 0.3, 0.2])
departments = np.random.choice(['Sales', 'HR', 'IT'], size=n_samples)

education_multiplier = {'Bachelor': 1.0, 'Master': 1.1, 'PhD': 1.2}
dept_multiplier_dict = {'Sales': 1.0, 'HR': 0.9, 'IT': 1.1}

# Construct the target Salary
base_salary = 20000
coef_exp_linear = 1000
coef_exp_quad = 200
coef_age = 100
noise = np.random.normal(0, 2000, n_samples)

salary = (
    base_salary +
    coef_exp_linear * years_experience +
    coef_exp_quad * (years_experience ** 2) +
    coef_age * age +
    bonus
)

# Apply the education multiplier
edu_multiplier = np.array([education_multiplier[edu] for edu in education_levels])
salary = salary * edu_multiplier + noise

# Apply the department multiplier
# TODO: Opravte logickou chybu v následujících dvou řádcích kódu!
dept_multiplier = np.array([education_multiplier[edu] for edu in education_levels])
salary = salary * edu_multiplier + noise

# Create a DataFrame
data = pd.DataFrame({
    'YearsExperience': years_experience,
    'Age': age,
    'Bonus': bonus,
    'Education': education_levels,
    'Department': departments,
    'Salary': salary
})

# ----- Introduce Random NaN Values -----
nan_fraction = 0.1  # 10% missing values
for col in ['YearsExperience', 'Age', 'Bonus', 'Education', 'Department']:
    n_missing = int(n_samples * nan_fraction)
    missing_indices = np.random.choice(n_samples, size=n_missing, replace=False)
    data.loc[missing_indices, col] = np.nan

data.head()

## 5. Aktivní úkoly

Nyní začíná vaše samostatná práce. Vaším úkolem je vytvořit end-to-end model nad výše vygenerovanými daty.

### Úkol 1 a 2: Záchrana datové struktury a Prevence Data Leakage
1. Extrahovaná matice příznaků `X` nesmí ztratit formát `DataFrame`. Upravte kód tak, aby odstranil `.values` a využil metodu `data.drop()`.
2. Pokud byste začali doplňovat chybějící data (imputovat) ještě před rozdělením datasetu, dopustili byste se úniku informací (Data Leakage). Proveďte rozdělení dat pomocí `train_test_split` rovnou v této buňce.


In [ ]:
from sklearn.model_selection import train_test_split

# TODO 1: Opravte extrakci X, aby zůstalo jako DataFrame
X = data.iloc[:, :-1].values
y = data.iloc[:, -1].values

# TODO 2: Proveďte rozdělení dat (test_size=0.2, random_state=42)
# X_train, X_test, y_train, y_test = ...

### Úkol 3: Sestavení ColumnTransformeru (Předzpracování)
Nemůžete použít jeden `SimpleImputer` na celou matici `X`, protože obsahuje čísla i text.

1. Vytvořte logiku pro **numerické příznaky** (`YearsExperience`, `Age`, `Bonus`): Doplňte chybějící hodnoty mediánem a následně aplikujte `StandardScaler`.
2. Vytvořte logiku pro **kategoriální příznaky** (`Education`, `Department`): Doplňte chybějící hodnoty nejčastější hodnotou (`most_frequent`) a aplikujte `OneHotEncoder(handle_unknown='ignore')`.
3. Spojte obě tyto logiky do jednoho objektu `ColumnTransformer`.


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 1. Definice sloupců
num_features = ['YearsExperience', 'Age', 'Bonus']
cat_features = ['Education', 'Department']

# 2. Pipeline pro numerické příznaky
num_transformer = Pipeline(steps=[
    # TODO: Doplňte SimpleImputer(strategy=...)
    # TODO: Doplňte StandardScaler()
])

# 3. Pipeline pro kategoriální příznaky
cat_transformer = Pipeline(steps=[
    # TODO: Doplňte SimpleImputer(strategy=...)
    # TODO: Doplňte OneHotEncoder(...)
])

# 4. Spojení obou větví
# TODO: Definujte ColumnTransformer
# preprocessor = ColumnTransformer(transformers=[...])

### Úkol 4: Kompletní Pipeline a Evaluace
Sestavte finální model. Vytvořte `Pipeline`, která jako první krok přijme váš `preprocessor`, jako druhý krok provede transformaci `PolynomialFeatures(degree=2)` a nakonec aplikuje `LinearRegression()`.

Tuto pipeline natrénujte metodou `.fit()` **pouze na trénovacích datech**. Následně vygenerujte predikce pro testovací data a vypočítejte RMSE.


In [ ]:
# TODO: Sestavte finální model
# model = Pipeline(steps=[...])

# TODO: Natrénujte model POUZE na X_train, y_train
# ...

# TODO: Vygenerujte predikce na testovací sadě
# y_pred = ...

# TODO: Vypočítejte a vypište RMSE
# rmse = root_mean_squared_error(y_test, y_pred)
# print(f"RMSE na testovací sadě: {rmse}")


## 6. Experiment a Samostatný úkol

**Experiment:**
Vraťte se do buňky s 1D ukázkou na začátku notebooku. Změňte `degree = 3` na `degree = 1` a buňku spusťte. Následně vyzkoušejte `degree = 20`.
Který z těchto stavů ilustruje nedoučení (underfitting) a který přeučení (overfitting)?

**Samostatný úkol pro pokročilé:**
V Úkolu 4 jsme aplikovali `PolynomialFeatures` na celou šířku matice, tedy i na binární (0/1) sloupce, které vygeneroval One-Hot Encoding. Umocňovat 0 nebo 1 na druhou ale nedává matematický smysl a neúměrně zvětšuje datovou sadu.
*Úkol:* Upravte svůj kód tak, abyste krok `PolynomialFeatures` přesunuli z hlavní modelové Pipeline přímo dovnitř `num_transformer` větve. Tím se polynomiálně umocní pouze čísla a až poté se spojí s kategoriemi.

## 7. Kontrolní otázky (závěrečná sebereflexe)
1. Proč voláme na testovací sadě při generování predikcí pouze metodu `.transform()` (potažmo `.predict()`) a nikdy `.fit_transform()`?
2. Pokud proměnná `Education` obsahuje 3 unikátní hodnoty a `Department` také 3 unikátní hodnoty, kolik nových sloupců encodér do naší matice celkově přidá?
3. Šum generovaný v naší tabulce v kroku B měl směrodatnou odchylku 2000. Pokud vaše výsledné RMSE vyšlo poblíž této hodnoty, proč můžeme prohlásit náš model za excelentní?

---
### 7.1 Krátká reflexe na cvičení - díky za pomoc!

Který úkol nebo koncept pro vás byl dnes nejjednodušší? **[ Sem zadejte svou odpověď ]**

Po zodpovězení všech otázek v návaznosti na kód, vnímate že chápete jak funguje Simple Reflex Agent, k čemu se hodí a jakéí jsou jeho omezení v porovnání k Model-Based Agent? **[ Sem zadejte svou odpověď ]**

